<a href="https://colab.research.google.com/github/MinaAlberDS/Codveda-Internship/blob/master/Level%201/Task%201/Web_scrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scrapping, and collecting the data

### Collecting the Real Estates links

In [1]:
# Import the needed libraries
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
# Make the function which will get the real_estates links
def real_estates_links(pages:int,verbose:bool=True):
    """
    Scrapes real estate listing links from real estate ate.gov.eg.

    Args:
        pages: An integer representing the number of pages to scrape.
        verbose: A boolean indicating whether to print the progress of scraping each page.

    Returns:
        A list of strings, where each string is a URL to a real estate listing.
        :param pages: An integer representing the number of pages to scrape.
        :param verbose: A boolean indicating whether to print the progress of scraping each page.

    """
    base_url = r"https://realestate.gov.eg/ar/search/%D8%B4%D8%B1%D8%A7%D8%A1?page={}"
    links_list = [] # links list
    for page in range(1,pages+1): # This for loop to get the 9 real_estates links in each page
        start_time = time.perf_counter() # start a stop watch for each page
        page_url = base_url.format(page) # Set the page url from the baseline url using f-string or format
        response = requests.get(page_url) # Get the response of the page url
        if response.status_code == 200: # check if the page can be scrapped or not
          soap = BeautifulSoup(response.content, "html.parser") # set the soop variable to scrape the links from the page html
          real_estates = soap.find_all("div", class_="col-md-4")[3:] # get the real_estates links

          for real_estate in real_estates: # Add the real_estates links to the links list
              link = r"https://realestate.gov.eg" + real_estate.find("a")["href"] # Set the real_estate url
              links_list.append(link) # add it to the links list
          end_time = time.perf_counter() # stop the timer
          elapsed = end_time - start_time # count the elapsed time
          if verbose: # if the verbose is true, it will show some useful information about each page
              print(f"Page: {page} links have been appended in {elapsed:.4f}s")
        else: # if the page cannot be scraped
          print(f"failed to scrape this Page: {page} with code: {response.status_code}")
        time.sleep(0.5)
    return links_list # finally retrun the links list

In [2]:
links = real_estates_links(50, True) # running our function with 350 pages, and verbose = True

Page: 1 links have been appended in 2.5953s
Page: 2 links have been appended in 2.4979s
Page: 3 links have been appended in 3.0739s
Page: 4 links have been appended in 2.4732s
Page: 5 links have been appended in 2.4727s
Page: 6 links have been appended in 2.5073s
Page: 7 links have been appended in 2.5300s
Page: 8 links have been appended in 2.4963s
Page: 9 links have been appended in 2.5290s
Page: 10 links have been appended in 2.5319s


KeyboardInterrupt: 

In [ ]:
len(links)

450

|We have 504 real_estate to scrape

In [ ]:
import json
# export the real_estates links to a json file
with open("../data/raw/Links.json", "w") as links_json:
    json.dump(links, links_json, indent =4)

In [ ]:
# import the json file links to scrap them
with open("../data/raw/Links.json", "r") as links_json:
  links = json.load(links_json)
# show the results
links[:5]

['https://realestate.gov.eg/ar/property-details/mamsha-avenue/new-administrative-capital/cairo/building-maa15b-floor-1-unit-maa15b14-mamsha-avenue-new-administrative-capital-cairo-cairo-egypt/E189166',
 'https://realestate.gov.eg/ar/property-details/new-garden-city/new-administrative-capital/cairo/building-g883-floor-mezzanine-unit-g883mr10-new-garden-city-new-administrative-capital-cairo-cairo-egypt/E377964',
 'https://realestate.gov.eg/ar/property-details/maspero-mall/cairo/cairo/building-mm-floor-ground-floor-unit-mmrtg24b-maspero-mall-cairo-cairo-cairo-egypt/E110528',
 'https://realestate.gov.eg/ar/property-details/alamain-(latin-district)/north-coast/alexandria/building-a03-floor-3-unit-z05-cl08-a03-x4-03-05-alamain-(latin-district)-north-coast-alexandria-egypt/E375962',
 'https://realestate.gov.eg/ar/property-details/zahya/mansoura/dakahlia/building-zh3m71-unit-zh3m71-zahya-mansoura-(m)-dakahlia-egypt/E110258']

### Collect the real_estates data

In [ ]:
import json

def collect_real_estate_data(real_estates_links:list, verbose:bool=True, except_2025:bool=False):
    """
    Scrapes real estate data from a list of real estate listing URLs.
    Args:
        real_estates_links: A list of strings, where each string is a URL to a real estate listing.
        verbose: A boolean indicating whether to print the progress of scraping each real estate listing.
        except_2025: A boolean indicating whether to exclude listings built in 2025.
    Returns:
        A dictionary containing lists of scraped data for each real estate listing.
    """
    # Initialize a dictionary to store data for all real_estates
    all_real_estates_data = {
        'name': [],
        'price': [],
        'property_type': [],
        'bedrooms': [],
        'bathrooms': [],
        'sqm': [],
        'year_built': [],
        'features': [],
        'city': [],
        'governorate': [],
        'address': [],
        'description': []
    }

    # Open the JSON file in append mode outside the loop
    with open("../data/processed/real_estates_data.json", "w") as real_estates_data:
        # Initialize the JSON file with an empty list
        json.dump([], real_estates_data)


    for i, real_estate_link in enumerate(real_estates_links): # get the data from each real_estate
        try:
            real_estate_response = requests.get(real_estate_link) # Get the request of the real_estate link
            real_estate_soop = BeautifulSoup(real_estate_response.content, "html.parser") # make the real_estate soop

            real_estate_div = real_estate_soop.find_all("div", class_= "propertySpecs_spec_box__xLU5D")
            if len(real_estate_div) > 0:
                year_built = real_estate_div[4].find_all("span")[1]

                if except_2025 and year_built and year_built.text.strip() not in ["2022","2023", "2024"]:
                    if verbose:
                        print(f"Skipping real_estate {i+1}/{len(real_estates_links)} built in 2025")
                    continue  # Skip this real_estate if it's built in 2025

                # Extract data, handling potential missing elements
                name = real_estate_soop.find("h6", class_= "propertyInfo_name__bs0i7")
                all_real_estates_data['name'].append(name.get_text(strip=True) if name else None)

                price = real_estate_soop.find("p", class_ = "propertyInfo_price__8ecPp")
                all_real_estates_data['price'].append(price.text.strip() if price else None)

                property_type = real_estate_div[0].find("span")
                all_real_estates_data['property_type'].append(property_type.text.strip() if property_type else None)

                # Extract bedroom, bathroom, sqm, and year built if available

                bedroom = real_estate_div[1].find_all("span")[1]
                all_real_estates_data['bedrooms'].append(bedroom.text.strip() if bedroom else None)
                bathroom = real_estate_div[2].find_all("span")[1]
                all_real_estates_data['bathrooms'].append(bathroom.text.strip() if bathroom else None)
                sqm = real_estate_div[3].find_all("span")[1]
                all_real_estates_data['sqm'].append(sqm.text.strip() if sqm else None)
                all_real_estates_data['year_built'].append(year_built.text.strip() if year_built else None)

            else:
                # Append None for property type and other specs if the main div is not found
                all_real_estates_data['property_type'].append(None)
                all_real_estates_data['bedrooms'].append(None)
                all_real_estates_data['bathrooms'].append(None)
                all_real_estates_data['sqm'].append(None)
                all_real_estates_data['year_built'].append(None)


            real_estate_features = []
            real_estate_features_dv = real_estate_soop.find_all("div", class_ = "col-md-6")
            for real_estate_feature in real_estate_features_dv:
                feature_text = real_estate_feature.find("h2")
                if feature_text:
                    real_estate_features.append(feature_text.get_text(strip=True))
            all_real_estates_data['features'].append(real_estate_features) # Append the list of features


            real_estate_city_div = real_estate_soop.find_all("div", class_= "propertyData_address_info_row__up8MX")
            if len(real_estate_city_div) > 2:
                city = real_estate_city_div[1].find_all("p")
                all_real_estates_data['city'].append(city[1].get_text(strip=True) if len(city) > 1 else None)

                governorate = real_estate_city_div[2].find_all("p")
                all_real_estates_data['governorate'].append(governorate[1].get_text(strip=True) if len(governorate) > 1 else None)
            else:
                # Append None if city and governorate details are not found
                all_real_estates_data['city'].append(None)
                all_real_estates_data['governorate'].append(None)


            address_div = real_estate_soop.find("div", class_="propertyData_address_info_row__up8MX propertyData_address_row__vZiYP")
            if address_div:
                address = address_div.find_all("p")
                all_real_estates_data['address'].append(address[1].get_text(strip=True) if len(address) > 1 else None)
            else:
                # Append None if address is not found
                all_real_estates_data['address'].append(None)

            description_div = real_estate_soop.find("div", class_= "propertyData_property_long_description__mAJPf")
            if description_div:
                 # Extract text from each paragraph tag and join them
                 paragraph_texts = [p.get_text(strip=True) for p in description_div.find_all("p")]
                 full_description = "".join(paragraph_texts)
                 all_real_estates_data['description'].append(full_description)
            else:
                # Append None if description is not found
                all_real_estates_data['description'].append(None)


            # Append the data of the current real_estate to the JSON file
            with open("../data/processed/real_estates_data.json", "r+") as real_estates_data:
                data = json.load(real_estates_data)
                data.append({key: all_real_estates_data[key][-1] for key in all_real_estates_data})  # Append the last added data
                real_estates_data.seek(0) # the begining of the file
                json.dump(data, real_estates_data, indent=4) # Add the real_estate data
                real_estates_data.truncate() # The beining of the file

            if verbose:
                print(f"Scraped data for real_estate {i+1}/{len(real_estates_links)}")

        except Exception as e:
            print(f"Error scraping {real_estate_link}: {e}")
            # Append None for all fields if an error occurs during scraping
            for key in all_real_estates_data:
                all_real_estates_data[key].append(None)

        time.sleep(0.5)
    return all_real_estates_data

#### Except 2025 real_estates

In [ ]:
real_estates_data = collect_real_estate_data(links, except_2025=True)
real_estates_data

Skipping real_estate 1/450 built in 2025
Skipping real_estate 2/450 built in 2025
Scraped data for real_estate 3/450
Skipping real_estate 4/450 built in 2025
Scraped data for real_estate 5/450
Skipping real_estate 6/450 built in 2025
Skipping real_estate 7/450 built in 2025
Skipping real_estate 8/450 built in 2025
Skipping real_estate 9/450 built in 2025
Skipping real_estate 10/450 built in 2025
Scraped data for real_estate 11/450
Scraped data for real_estate 12/450
Scraped data for real_estate 13/450
Skipping real_estate 14/450 built in 2025
Skipping real_estate 15/450 built in 2025
Skipping real_estate 16/450 built in 2025
Skipping real_estate 17/450 built in 2025
Skipping real_estate 18/450 built in 2025
Skipping real_estate 19/450 built in 2025
Skipping real_estate 20/450 built in 2025
Skipping real_estate 21/450 built in 2025
Skipping real_estate 22/450 built in 2025
Skipping real_estate 23/450 built in 2025
Skipping real_estate 24/450 built in 2025
Scraped data for real_estate 25

{'name': ['Maspero mall',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Maspero Business Tower Nile Heights',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Maspero Business Tower Nile Heights',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Maspero Business Tower Nile Heights',
  'Maspero Business Tower Nile Heights',
  'Zahya',
  'Zahya',
  'Zahya',
  'Maspero mall',
  'Maspero Business Tower Nile Heights',
  'Zahya',
  'Zahya',
  'Maspero Business Tower Nile Heights',
  'Zahya',
  'Maspero Business Tower Nile Heights',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Maspero Business Tower Nile Heights',
  'Maspero Business Tower Nile Heights',
  'Zahya',
  'Zahya',
  'Maspero Business Tower Nile Heights',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Zahya',
  'Maspero Business Tower Nil

### Include 2025 real_estates

In [ ]:
# real_estates_data = collect_real_estate_data(links)
# real_estates_data

## Include 2025

### Exporting the data

In [ ]:
# real_estates_df = pd.read_json(r"../data/processed/real_estates_data.json")
# real_estates_df # convert to a dataset

,name,price,property_type,bedrooms,bathrooms,sqm,year_built,features,city,governorate,address,description
0,Maspero mall,"17,886,000 ج م","تجاري, مبنى البيع بالتجزئة / مول",0,0,117.0,2023,"[أخرى, أمن 24 ساعة, الكهرباء متاحة]",Cairo,Cairo,Building MM Floor Ground floor Unit MMRTG24B M...,يُعد ماسبيرو مول وجهة فريدة لتجارب التسوق ونمط...
1,Zahya,"17,536,000 ج م","سكني, فيلا",3,2,600.2,2024,"[اللوبي, أمن 24 ساعة, الكهرباء متاحة]",Mansoura,Dakahlia,Building ZH3M71 Unit ZH3M71 Zahya Mansoura (m)...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
2,Zahya,"22,024,000 ج م","سكني, فيلا",3,0,891.0,2024,"[اللوبي, أمن 24 ساعة, الكهرباء متاحة]",Mansoura,Dakahlia,Building ZH1C69 Floor Ground floor Unit ZH1C69...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
3,Zahya,"10,249,000 ج م","سكني, توين هاوس",3,0,350.5,2024,"[اللوبي, أمن 24 ساعة, الكهرباء متاحة]",Mansoura,Dakahlia,Building ZHM219 Floor Ground floor Unit ZHM219...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
4,Zahya,"17,478,000 ج م","سكني, فيلا",3,2,600.2,2024,"[اللوبي, أمن 24 ساعة, الكهرباء متاحة]",Mansoura,Dakahlia,Building ZH3M72 Unit ZH3M72 Zahya Mansoura (m)...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
...,...,...,...,...,...,...,...,...,...,...,...,...
61,Zahya,"10,612,000 ج م","سكني, توين هاوس",3,0,410.0,2024,"[اللوبي, أمن 24 ساعة, الكهرباء متاحة]",Mansoura,Dakahlia,Building ZH3M217 Unit ZH3M217B Zahya Mansoura ...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
62,Zahya,"10,209,000 ج م","سكني, توين هاوس",3,0,345.0,2024,"[اللوبي, أمن 24 ساعة, الكهرباء متاحة]",Mansoura,Dakahlia,Building ZHM219 Floor Ground floor Unit ZHM219...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
63,Zahya,"10,669,000 ج م","سكني, توين هاوس",3,0,410.0,2024,"[اللوبي, أمن 24 ساعة, الكهرباء متاحة]",Mansoura,Dakahlia,Building ZH3M226 Unit ZH3M226A Zahya Mansoura ...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
64,Maspero Business Tower Nile Heights,"12,026,000 ج م","تجاري, مكتب",0,0,185.0,2023,"[أخرى, أمن 24 ساعة, مجتمع مسور, الكهرباء متاحة]",Cairo,Cairo,Building NH Floor 2nd floor Unit NHAD0201 Masp...,من الإطلالات الخلابة على أفق القاهرة المتألّق ...


In [ ]:
# #Export it as a csv file
# real_estates_df.to_csv("../data/processed/real_estates_data.csv", index=False)

I scrapped more 283 rows, so the final output is 1183 rows

In [ ]:
# re_df = pd.read_csv(r"../data/processed/old_real_estates.csv")
# re_df_77 = pd.read_csv(r"../data/processed/real_estates_data(1).csv")
# re_df_206 = pd.read_csv(r"../data/processed/real_estates_data(2).csv")
#
# # Concatenate all dataframes vertically (row-wise)
# re_df = pd.concat([re_df, re_df_77, re_df_206], ignore_index=True)
#
# re_df

,name,price,property_type,bedrooms,bathrooms,sqm,year_built,features,city,governorate,address,description
0,Maspero mall,"17,886,000 ج م","تجاري, مبنى البيع بالتجزئة / مول",0,0,117.00,2023,"['أخرى', 'أمن 24 ساعة', 'الكهرباء متاحة']",Cairo,Cairo,Building MM Floor Ground floor Unit MMRTG24B M...,يُعد ماسبيرو مول وجهة فريدة لتجارب التسوق ونمط...
1,Zahya,"17,536,000 ج م","سكني, فيلا",3,2,600.20,2024,"['اللوبي', 'أمن 24 ساعة', 'الكهرباء متاحة']",Mansoura,Dakahlia,Building ZH3M71 Unit ZH3M71 Zahya Mansoura (m)...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
2,Zahya,"22,024,000 ج م","سكني, فيلا",3,0,891.00,2024,"['اللوبي', 'أمن 24 ساعة', 'الكهرباء متاحة']",Mansoura,Dakahlia,Building ZH1C69 Floor Ground floor Unit ZH1C69...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
3,Zahya,"10,249,000 ج م","سكني, توين هاوس",3,0,350.50,2024,"['اللوبي', 'أمن 24 ساعة', 'الكهرباء متاحة']",Mansoura,Dakahlia,Building ZHM219 Floor Ground floor Unit ZHM219...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
4,Zahya,"17,478,000 ج م","سكني, فيلا",3,2,600.20,2024,"['اللوبي', 'أمن 24 ساعة', 'الكهرباء متاحة']",Mansoura,Dakahlia,Building ZH3M72 Unit ZH3M72 Zahya Mansoura (m)...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
...,...,...,...,...,...,...,...,...,...,...,...,...
344,T- Residences,"2,668,259 EGP","Residential, Apartment",3,1,120.00,2024,"['24 Hour Security', 'Electricity Available', ...",North Coast,Matrouh,Building C-12 Floor 6 Unit 62 T- Residences No...,"""Torec Developments is a subsidiary of the New..."
345,Alamain (Latin District),"3,226,000 EGP","Residential, Apartment",1,0,94.61,2025,"['Other', 'Security Gate', 'Electricity Availa...",North Coast,Alexandria,Building D15 Floor 7 Unit Z02-CL07-D15-X7-07-0...,Latini by Saudi Egyptian Developers (SED) – Ne...
346,Downtown commercial,"20,540,000 EGP","Commercial, Retail",0,0,200.00,2025,"['Other', '24 Hour Security', 'Electricity Ava...",Mersa Matruh,Matrouh,Building DT01 Floor Ground floor Unit DTRG0130...,Designed to be an integrated residential and c...
347,Alamain (Latin District),"6,512,000 EGP","Residential, Apartment",4,0,213.53,2025,"['Other', 'Security Gate', 'Electricity Availa...",North Coast,Alexandria,Building C06 Floor 1 Unit Z05-CL11-C06-M7-01-0...,Latini by Saudi Egyptian Developers (SED) – Ne...


In [ ]:
# re_df.to_csv("../data/processed/real_estates_data(new).csv", index=False)

## Except 2025

In [ ]:
re_df = pd.read_csv(r"../data/processed/real_estates_data(new).csv")
re_df_except_2025 = pd.read_csv(r"../data/processed/old_real_estates.csv")

re_df = pd.concat([re_df, re_df_except_2025], ignore_index=True)

re_df

,name,price,property_type,bedrooms,bathrooms,sqm,year_built,features,city,governorate,address,description
0,Alamain (Latin District),"6,594,000 EGP","Residential, Apartment",1,0,92.86,2025,"['Other', 'Security Gate', 'Electricity Availa...",North Coast,Alexandria,Building F04 Floor 6 Unit Z02-CL05-F04-X7-06-1...,Latini by Saudi Egyptian Developers (SED) – Ne...
1,Beachfront Tower - B1,"66,104,000 EGP","Residential, Apartment",3,3,398.00,2025,['Other'],Mersa Matruh,Matrouh,Building BTB1 Floor 9th floor Unit BTB1091 Bea...,Explore a lifestyle that allows you to have yo...
2,PODIA,"12,824,000 EGP","Commercial, Office",0,0,93.00,2025,"['Other', '24 Hour Security', 'Fire Alarm', 'G...",Cairo,Cairo,Floor 13th Unit MPO-1304-A Bin ZayedNorth PODI...,Launched by Menassat Developments in cooperati...
3,Mazarine Apartment,"12,857,000 EGP","Residential, Apartment",3,3,252.00,2025,"['24 Hour Security', 'Electricity Available']",Mersa Matruh,Matrouh,Building MZCS15 Floor 3rd floor Unit MZCS1531 ...,The name ‘Mazarine’ came to fruition based on ...
4,Alamain (Latin District),"8,721,000 EGP","Residential, Apartment",3,0,209.42,2025,"['Other', 'Security Gate', 'Electricity Availa...",North Coast,Alexandria,Building E01 Floor 1 Unit Z02-CL09-E01-X4-01-0...,One of the main districts in this new city nei...
...,...,...,...,...,...,...,...,...,...,...,...,...
1244,Zahya,"10,612,000 ج م","سكني, توين هاوس",3,0,410.00,2024,"['اللوبي', 'أمن 24 ساعة', 'الكهرباء متاحة']",Mansoura,Dakahlia,Building ZH3M217 Unit ZH3M217B Zahya Mansoura ...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
1245,Zahya,"10,209,000 ج م","سكني, توين هاوس",3,0,345.00,2024,"['اللوبي', 'أمن 24 ساعة', 'الكهرباء متاحة']",Mansoura,Dakahlia,Building ZHM219 Floor Ground floor Unit ZHM219...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
1246,Zahya,"10,669,000 ج م","سكني, توين هاوس",3,0,410.00,2024,"['اللوبي', 'أمن 24 ساعة', 'الكهرباء متاحة']",Mansoura,Dakahlia,Building ZH3M226 Unit ZH3M226A Zahya Mansoura ...,بصفتنا شركة عقارية رائدة تُعنى ببناء مشروع سكن...
1247,Maspero Business Tower Nile Heights,"12,026,000 ج م","تجاري, مكتب",0,0,185.00,2023,"['أخرى', 'أمن 24 ساعة', 'مجتمع مسور', 'الكهربا...",Cairo,Cairo,Building NH Floor 2nd floor Unit NHAD0201 Masp...,من الإطلالات الخلابة على أفق القاهرة المتألّق ...


In [ ]:
re_df.to_csv("../data/processed/real_estates_data_except_2025.csv", index=False)